### Version 3 that adds region of teams, tournaments, dates, core being together.
Still missing : 
- core being together
- region of teams
- formating of the date (there are lists and the datetime library that makes a str date)

#### BO3 handling

##### Make a dataframe of the matches that have happened 

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

from scipy.stats import gaussian_kde
import numpy as np
from datetime import datetime

# Function to extract match data from a single page
def extract_matches(soup):
    data = []
    for sublist in soup.find_all("div", class_="results-sublist"):
        date_element = sublist.find("span", class_="standard-headline")
        if date_element:
            date = date_element.text.strip().split()[-3:] # the str is "Results for Month day year" therefore take only the last 3 elements of the list
        else:
            date = datetime.now().strftime("%B %d, %Y")  # Assign today's date when date is None
        for result_con in sublist.find_all("div", class_="result-con"):
            a_reset = result_con.find("a", class_="a-reset")
            if a_reset:
                result = a_reset.find("div", class_="result")

                # Extract team names, score, and match link
                team1 = result.find_all("td", class_="team-cell")[0].text.strip()
                score = result.find("td", class_="result-score").text.strip()
                team2 = result.find_all("td", class_="team-cell")[1].text.strip()
                match_link = "https://www.hltv.org" + a_reset["href"]
                event = result.find("td",class_="event")
                if event:
                    event_name = result.find("span",class_='event-name').text.strip()

                # Create a dictionary for each match with relevant data
                match_data = {
                    "Team 1": team1,
                    "Team 2": team2,
                    "Score": score,
                    "Match Link": match_link,
                    "Event Name": event_name,
                    "Date": date,
                }
                data.append(match_data)

    return data

# Set up the WebDriver (without specifying the driver path)
driver = webdriver.Firefox()

# Navigate to the first page
url = "https://www.hltv.org/results"
driver.get(url)

# Wait for the cookie banner to appear (up to 10 seconds) and click it
try:
    wait = WebDriverWait(driver, 10)
    cookie_banner = wait.until(EC.presence_of_element_located((By.XPATH, "//*[@id='CybotCookiebotDialogBodyButtonDecline']")))
    cookie_banner.click()
except TimeoutException:
    print("Cookie banner not found. Continuing with scraping...")

# Scrape match history for a specified number of pages
num_pages = 18  # Change this value to the desired number of pages
match_history = []

for page in range(1, num_pages + 1):
    offset = (page - 1) * 100
    url = f"https://www.hltv.org/results?offset={offset}"

    driver.get(url)

    # Get the HTML content
    html_content = driver.page_source

    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")

    match_history.extend(extract_matches(soup)[:])

# Close the WebDriver
driver.quit()

# Create a pandas DataFrame with the scraped data
df = pd.DataFrame(match_history)

# Save the DataFrame to an Excel file
df.to_excel("match_history.xlsx", index=False)

# Look at winrates by adding a column of who won
# Step 1: Create a new column 'Outcome'
def get_winner(score, team1, team2):
    s1, s2 = map(int, score.split(" - "))
    if s1 > s2:
        return team1
    if s1 < s2:
        return team2
    else:
        return "Draw"

df["Outcome"] = df[["Score", "Team 1", "Team 2"]].apply(lambda row: get_winner(row["Score"], row["Team 1"], row["Team 2"]), axis=1)

# Step 2: Create a new DataFrame 'bo1s' for rows with a score superior to 3 to account for matches that are either BO1 or BO5
def is_score_superior_to_3(score):
    s1, s2 = map(int, score.split(" - "))
    return s1 > 3 or s2 > 3

bo1s = df[df["Score"].apply(is_score_superior_to_3)].copy()
df = df[~df["Score"].apply(is_score_superior_to_3)]
df

##### 2 functions to extract map names and rounds

In [ ]:
import sys
import time
def extract_map_names(match_link):
    print(f"Processing match link: {match_link}")

    # Set up the WebDriver (without specifying the driver path)
    driver = webdriver.Firefox()

    # Navigate to the match page
    driver.get(match_link)

    try:
        # Wait for the flexbox-column element to appear (up to 10 seconds)
        wait = WebDriverWait(driver, 10)
        flexbox_column = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.flexbox-column")))

        print("Found flexbox-column")

        map_names = []

        # Step 2: Iterate through the child elements with <div class="mapholder">
        for mapholder in flexbox_column.find_elements(By.CSS_SELECTOR, "div.mapholder"):
            print("Found mapholder")
            # Step 3: For each mapholder, find the child element with <div class="played">
            played = mapholder.find_element(By.CSS_SELECTOR, "div.played")

            if played:
                print("Found played")
                # Step 4: Inside the played element, find the child element with <div class="map-name-holder">
                map_name_holder = played.find_element(By.CSS_SELECTOR, "div.map-name-holder")

                if map_name_holder:
                    print("Found map-name-holder")
                    # Step 5: Extract the map name from the map-name-holder element
                    map_name_element = map_name_holder.find_element(By.CSS_SELECTOR, "div.mapname")

                    if map_name_element:
                        print("Found mapname")
                        map_name = map_name_element.text
                        map_names.append(map_name)
                        print(f"Appended map name: {map_name}")                
            print("Found")
    except NoSuchElementException:
    # Handle "Are you a robot?" page
        try:
            
            robot_button = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="CybotCookiebotDialogBodyButtonDecline"]')))
            print("Located 'I am not a robot' button")
            #CybotCookiebotDialogBodyButtonDecline
            robot_button.click()
            print("Clicked 'I am not a robot' button")

        except Exception as e:
            print(f"Error while clicking 'I am not a robot' button: {match_link}")
    except Exception as e:
        print(f"Error while processing match link: {match_link} map names")
        #sys.exit()
    #finally:
        # Close the WebDriver
    driver.quit()

    return map_names

In [ ]:
def extract_map_data(match_link):
    print(f"Processing match link: {match_link}")

    # Set up the WebDriver (without specifying the driver path)
    driver = webdriver.Firefox()

    # Navigate to the match page
    driver.get(match_link)

    try:
        # Wait for the flexbox-column element to appear (up to 10 seconds)
        wait = WebDriverWait(driver, 10)
        flexbox_column = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.flexbox-column")))

        print("Found flexbox-column")

        map_data = []
        # Step 1: Extract the map names
        map_names = extract_map_names(match_link)

        # Step 2: Iterate through the child elements with <div class="mapholder">
        mapholder_index = 1
        for mapholder in flexbox_column.find_elements(By.CSS_SELECTOR, "div.mapholder"):
            print("Found mapholder")

            # Get the map name for the current mapholder
            map_name = map_names[mapholder_index - 1]

            # Wait for the score elements to appear (up to 10 seconds)
            team1_score_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'div.mapholder:nth-child({mapholder_index}) div.results.played div[class^="results-left"] div.results-team-score')))
            team2_score_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'div.mapholder:nth-child({mapholder_index}) div.results.played span[class^="results-right"] div.results-team-score')))

            # Extract the scores (text from the elements)
            team1_score = int(team1_score_element.text)
            team2_score = int(team2_score_element.text)

            # Create a dictionary with the map data
            map_data.append({
                "map_name": map_name,
                "team1_score": team1_score,
                "team2_score": team2_score
            })

            print(f"Appended map data: {map_data[-1]}")

            mapholder_index += 1
    except NoSuchElementException:
        webdriver.wait
    # Handle "Are you a robot?" page
        try:
            time.sleep(7)
            robot_button = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="CybotCookiebotDialogBodyButtonDecline"]')))
            print("Located 'I am not a robot' button")
            #CybotCookiebotDialogBodyButtonDecline
            robot_button.click()
            print("Clicked 'I am not a robot' button")
        except Exception as e:
            print(f"Error while clicking 'I am not a robot' button: {e}")
                    
    except Exception as e:
        print(f"Error while processing match link: {match_link} map data")
        #sys.exit()
    #finally:
        # Close the WebDriver
    driver.quit()

    return map_data


##### Make a dataframe that uses both functions exract_map_data and extract_map_names

In [ ]:
# Assuming 'df' is your existing DataFrame with match links
df["Match Data"] = df["Match Link"].apply(lambda x: extract_map_data(x))

# Flatten the list of dictionaries in the "Match Data" column
match_data_flattened = [item for sublist in df["Match Data"] for item in sublist]

# Create a new DataFrame from the flattened list of dictionaries
match_data_df = pd.DataFrame(match_data_flattened)

# Add the new DataFrame to your existing DataFrame
df = pd.concat([df, match_data_df], axis=1)

# Iterate through the rows of the dataframe
for i, row in df.iterrows():
    # Extract the dictionary from the "Match Data" column
    match_data = row['Match Data']
    # If the value is a float, skip to the next row
    if isinstance(match_data, float):
        continue
    # Iterate through the dictionaries in the list
    for j, data in enumerate(match_data, start=1):
        # Extract the map name and scores for both teams
        map_name = data['map_name']
        team1_score = data['team1_score']
        team2_score = data['team2_score']
        # Create new columns in the dataframe for each map played
        map_col_name = f"map{j}"
        team1_col_name = f"team1_score_map{j}"
        team2_col_name = f"team2_score_map{j}"
        # Fill in the map name and team scores for the appropriate row
        df.loc[i, map_col_name] = map_name
        df.loc[i, team1_col_name] = team1_score
        df.loc[i, team2_col_name] = team2_score

df.drop(columns=["map_name", "team1_score", "team2_score"], inplace=True)
df.dropna(axis=0, how='all', inplace=True)


In [ ]:
df.to_excel("match_history_with_map_data.xlsx", index=False)

#### Get winrates for bo3 matches and bo1 matches

In [ ]:
df = pd.read_excel("match_history_with_map_data.xlsx")


In [ ]:
# create a new dataframe to store win rates
winrates_stats = pd.DataFrame(columns=['team_name', 'wins', 'defeats'])

# iterate through each row of the original dataframe
for i, row in df.iterrows():
    # get the team name from the 'Outcome' column
    team = row['Outcome']

    # check if the team already exists in the winrates_stats dataframe
    team_exists = winrates_stats['team_name'].isin([team]).any()

    # if the team doesn't exist, add a new row to the dataframe with their name and set wins and defeats to 0
    if not team_exists:
        new_row = pd.DataFrame([[team, 0, 0]], columns=['team_name', 'wins', 'defeats'])
        winrates_stats = pd.concat([winrates_stats, new_row], ignore_index=True)

    # find the index of the team in the winrates_stats dataframe
    team_index = winrates_stats[winrates_stats['team_name'] == team].index[0]

    # add 1 to the wins column for the team
    winrates_stats.at[team_index, 'wins'] += 1

    # count the number of times the team appears in either the 'Team 1' or 'Team 2' columns
    team_appearances = df[(df['Team 1'] == team) | (df['Team 2'] == team)]

    # subtract the number of wins from the total number of appearances to get the number of defeats
    defeats = team_appearances.shape[0] - winrates_stats.at[team_index, 'wins']

    # add the number of defeats to the defeats column for the team
    winrates_stats.at[team_index, 'defeats'] = defeats

    # calculate the overall win rate
winrates_stats['overall_winrate'] = (winrates_stats['wins'] / (winrates_stats['wins'] + winrates_stats['defeats']))*100

winrates_stats

In [ ]:
# create a new dataframe to store win rates for bo1s
winrates_stats2 = pd.DataFrame(columns=['team_name', 'wins', 'defeats'])

# iterate through each row of the original dataframe
for i, row in bo1s.iterrows():
    # get the team name from the 'Outcome' column
    team = row['Outcome']

    # check if the team already exists in the winrates_stats2 dataframe
    team_exists = winrates_stats2['team_name'].isin([team]).any()

    # if the team doesn't exist, add a new row to the dataframe with their name and set wins and defeats to 0
    if not team_exists:
        new_row = pd.DataFrame([[team, 0, 0]], columns=['team_name', 'wins', 'defeats'])
        winrates_stats2 = pd.concat([winrates_stats2, new_row], ignore_index=True)

    # find the index of the team in the winrates_stats2 dataframe
    team_index = winrates_stats2[winrates_stats2['team_name'] == team].index[0]

    # add 1 to the wins column for the team
    winrates_stats2.at[team_index, 'wins'] += 1

    # count the number of times the team appears in either the 'Team 1' or 'Team 2' columns
    team_appearances = bo1s[(bo1s['Team 1'] == team) | (bo1s['Team 2'] == team)]

    # subtract the number of wins from the total number of appearances to get the number of defeats
    defeats = team_appearances.shape[0] - winrates_stats2.at[team_index, 'wins']

    # add the number of defeats to the defeats column for the team
    winrates_stats2.at[team_index, 'defeats'] = defeats

    # calculate the overall win rate
winrates_stats2['overall_winrate'] = (winrates_stats2['wins'] / (winrates_stats2['wins'] + winrates_stats2['defeats']))*100

merged_df = winrates_stats.merge(winrates_stats2, how='left', on='team_name')
merged_df

Merge winrates of bo1s and bo3s into one table

In [ ]:
import numpy as np

# Add 'wins_y' to 'wins_x' when 'wins_y' is not NaN
merged_df['wins_x'] = np.where(merged_df['wins_y'].notna(), merged_df['wins_x'] + merged_df['wins_y'], merged_df['wins_x'])

# Add 'defeats_y' to 'defeats_x' when 'defeats_y' is not NaN
merged_df['defeats_x'] = np.where(merged_df['defeats_y'].notna(), merged_df['defeats_x'] + merged_df['defeats_y'], merged_df['defeats_x'])

# Drop unnecessary columns
merged_df = merged_df.drop(columns=['wins_y', 'defeats_y', 'overall_winrate_y'])

# Rename columns
merged_df = merged_df.rename(columns={'wins_x': 'wins', 'defeats_x': 'defeats', 'overall_winrate_x': 'overall_winrate'})
# calculate the overall win rate
merged_df['overall_winrate'] = (merged_df['wins'] / (merged_df['wins'] + merged_df['defeats']))*100

merged_df

#### Win rates per map and amount of rounds won per map

In [ ]:
# Initialize an empty dictionary
team_stats = {}

# Iterate over the rows of the dataframe
for index, row in df.iterrows():
    # Get the team names and scores
    teams = [row['Team 1'], row['Team 2']]
    scores = [[row['team1_score_map1'], row['team1_score_map2'], row['team1_score_map3']],
              [row['team2_score_map1'], row['team2_score_map2'], row['team2_score_map3']]]

    # Iterate over the teams
    for i in range(2):
        # If the team is not in the dictionary, add it
        if teams[i] not in team_stats:
            team_stats[teams[i]] = {'map1': {'map_name': row['map1'], 'rounds_won': []},
                                    'map2': {'map_name': row['map2'], 'rounds_won': []},
                                    'map3': {'map_name': row['map3'], 'rounds_won': []}}

        # Add the scores to the dictionary
        for j in range(3):
            if pd.notna(scores[i][j]):
                team_stats[teams[i]]['map' + str(j+1)]['rounds_won'].append(scores[i][j])

# Get all the unique map names
map_names = set()
for team in team_stats.values():
    for map_data in team.values():
        map_names.add(map_data['map_name'])

# Create a new dictionary with the team names and map names as keys
new_team_stats = {}
for team in team_stats.keys():
    new_team_stats[team] = {map_name: [] for map_name in map_names}

# Iterate through each dictionary and put the list that matches both the team name and the column that has the same map_name
for team, maps in team_stats.items():
    for map_num, map_data in maps.items():
        map_name = map_data['map_name']
        rounds_won = map_data['rounds_won']
        new_team_stats[team][map_name] += rounds_won

# Convert the dictionary to a dataframe
team_stats_df = pd.DataFrame(new_team_stats)
team_stats_df.T

In [ ]:
df


#### BO1 handling

In [ ]:
bo1s

In [ ]:
# Apply the extract_map_data function to the first num_rows rows
bo1s["Match Data"] = bo1s["Match Link"].apply(lambda x: extract_map_data(x))

# Flatten the list of dictionaries in the "Match Data" column
match_data_flattened = [item for sublist in bo1s["Match Data"] for item in sublist]

# Create a new DataFrame from the flattened list of dictionaries
match_data_df = pd.DataFrame(match_data_flattened)

# Add the new DataFrame to your existing DataFrame
bo1s = pd.concat([bo1s, match_data_df], axis=1)

# Iterate through the rows of the dataframe
for i, row in bo1s.iterrows():
    # Extract the dictionary from the "Match Data" column
    match_data = row['Match Data']
    # If the value is a float, skip to the next row
    if isinstance(match_data, float):
        continue
    # Iterate through the dictionaries in the list
    for j, data in enumerate(match_data, start=1):
        # Extract the map name and scores for both teams
        map_name = data['map_name']
        team1_score = data['team1_score']
        team2_score = data['team2_score']
        # Create new columns in the dataframe for each map played
        map_col_name = f"map{j}"
        team1_col_name = f"team1_score_map{j}"
        team2_col_name = f"team2_score_map{j}"
        # Fill in the map name and team scores for the appropriate row
        bo1s.loc[i, map_col_name] = map_name
        bo1s.loc[i, team1_col_name] = team1_score
        bo1s.loc[i, team2_col_name] = team2_score

bo1s.drop(columns=["map_name", "team1_score", "team2_score"], inplace=True)
bo1s.dropna(axis=0, how='all', inplace=True)
bo1s


In [ ]:
bo1s

In [ ]:
# Initialize an empty dictionary
team_stats = {}

# Iterate over the rows of the dataframe
for index, row in bo1s.iterrows():
    # Get the team names and scores
    teams = [row['Team 1'], row['Team 2']]
    scores = [[row['team1_score_map1'], row['team1_score_map2'], row['team1_score_map3']],
              [row['team2_score_map1'], row['team2_score_map2'], row['team2_score_map3']]]

    # Iterate over the teams
    for i in range(2):
        # If the team is not in the dictionary, add it
        if teams[i] not in team_stats:
            team_stats[teams[i]] = {'map1': {'map_name': row['map1'], 'rounds_won': []},
                                    'map2': {'map_name': row['map2'], 'rounds_won': []},
                                    'map3': {'map_name': row['map3'], 'rounds_won': []}}

        # Add the scores to the dictionary
        for j in range(3):
            if pd.notna(scores[i][j]):
                team_stats[teams[i]]['map' + str(j+1)]['rounds_won'].append(scores[i][j])

# Get all the unique map names
map_names = set()
for team in team_stats.values():
    for map_data in team.values():
        map_names.add(map_data['map_name'])

# Create a new dictionary with the team names and map names as keys
new_team_stats = {}
for team in team_stats.keys():
    new_team_stats[team] = {map_name: [] for map_name in map_names}

# Iterate through each dictionary and put the list that matches both the team name and the column that has the same map_name
for team, maps in team_stats.items():
    for map_num, map_data in maps.items():
        map_name = map_data['map_name']
        rounds_won = map_data['rounds_won']
        new_team_stats[team][map_name] += rounds_won

# Convert the dictionary to a dataframe
team_stats_bo1s = pd.DataFrame(new_team_stats)
team_stats_bo1s.T